In [67]:
# Importando as bibliotecas necesarias
import math
import random

In [68]:
# Definiendo la función de activación sigmoide
def sigmoid(x):
    # Definiendo un límite para evitar overflow en la función exponencial
    x = max(-500.0, min(500.0, x))
    # Retornando el valor de la función sigmoide
    return 1.0 / (1.0 + math.exp(-x))

In [69]:
# Definiendo la clase Layer para representar una capa de la red neuronal
class Layer:
    # Inicializando la capa con el número de entradas, neuronas, pesos y sesgos
    def __init__(self, n_inputs, n_neurons, weights=None, biases=None):
        # Si se proporcionan pesos, se utilizan, de lo contrario se generan aleatoriamente
        if weights is not None:
            self.weights = weights
        else:
            self.weights = [
                [random.uniform(-1, 1) for _ in range(n_inputs)]
                for _ in range(n_neurons)
            ]
        # Si se proporcionan sesgos, se utilizan, de lo contrario se inicializan a cero por cada neurona
        if biases is not None:
            self.biases = biases
        else:
            self.biases = [0.0] * n_neurons

    # Definiendo el método forward para calcular la salida de la capa dada una entrada
    def forward(self, inputs):
        # Guardando la entrada para su uso posterior
        self.last_input = inputs
        # Definiendo la salida de cada neurona en la capa
        self.last_output = []
        # Iterando sobre cada neurona para calcular su salida
        for neuron_idx in range(len(self.weights)):
            # Calculando la suma ponderada de las entradas para la neurona actual
            z = sum(
                w * x for w, x in zip(self.weights[neuron_idx], inputs)
            )
            # Agregando el sesgo a la suma ponderada
            z += self.biases[neuron_idx]
            # Aplicando la función de activación sigmoide a la suma ponderada y agregando el resultado a la salida de la capa
            self.last_output.append(sigmoid(z))
        # Retornando la salida de la capa
        return self.last_output

In [70]:
# Definiendo la clase Network para representar la red neuronal completa
class Network:
    # Inicializando la red con una lista de capas
    def __init__(self, layers):
        self.layers = layers

    # Definiendo el método forward para calcular la salida de la red dada una entrada
    def forward(self, inputs):
        current = inputs
        # Iterando sobre cada capa para calcular la salida de la red
        for layer in self.layers:
            current = layer.forward(current)
        # Retornando la salida final de la red
        return current
    # Definiendo el método count_parameters para contar el número total de parámetros (pesos y sesgos) en la red
    def count_parameters(self):
        total = 0
        # Iterando sobre cada capa para contar los pesos y sesgos
        for layer in self.layers:
            # Contando los pesos de cada neurona en la capa
            for neuron_weights in layer.weights:
                total += len(neuron_weights)
            # Contando los sesgos de cada neurona en la capa
            total += len(layer.biases)
        # Retornando el número total de parámetros en la red
        return total

In [71]:
print("=" * 60)
print("DEMO 1: XOR with hand-tuned 2-2-1 network")
print("=" * 60)

hidden = Layer(
    n_inputs=2,
    n_neurons=2,
    weights=[[20.0, 20.0], [-20.0, -20.0]],
    biases=[-10.0, 30.0],
)

output = Layer(
    n_inputs=2,
    n_neurons=1,
    weights=[[20.0, 20.0]],
    biases=[-30.0],
)

xor_net = Network([hidden, output])

xor_data = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

all_correct = True
for inputs, expected in xor_data:
    result = xor_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    status = "OK" if predicted == expected else "WRONG"
    if predicted != expected:
        all_correct = False
    print(f"  {inputs} -> {result[0]:.6f} (rounded: {predicted}, expected: {expected}) {status}")

print(f"\nXOR solved: {all_correct}")
print(f"Parameters: {xor_net.count_parameters()}")

DEMO 1: XOR with hand-tuned 2-2-1 network
  [0, 0] -> 0.000045 (rounded: 0, expected: 0) OK
  [0, 1] -> 0.999955 (rounded: 1, expected: 1) OK
  [1, 0] -> 0.999955 (rounded: 1, expected: 1) OK
  [1, 1] -> 0.000045 (rounded: 0, expected: 0) OK

XOR solved: True
Parameters: 9


In [72]:
print("=" * 60)
print("DEMO 2: Circle classification with 2-8-1 network")
print("=" * 60)

random.seed(42)

data = []
for _ in range(200):
    x = random.uniform(-1, 1)
    y = random.uniform(-1, 1)
    label = 1 if (x * x + y * y) < 0.25 else 0
    data.append(([x, y], label))

inside_count = sum(1 for _, label in data if label == 1)
outside_count = len(data) - inside_count
print(f"  Dataset: {len(data)} points ({inside_count} inside, {outside_count} outside)")

random.seed(7)
circle_net = Network([
    Layer(n_inputs=2, n_neurons=8),
    Layer(n_inputs=8, n_neurons=1),
])

correct = 0
for inputs, expected in data:
    result = circle_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    if predicted == expected:
        correct += 1

print(f"  Accuracy with random weights: {correct}/{len(data)} ({100 * correct / len(data):.1f}%)")
print(f"  Parameters: {circle_net.count_parameters()}")
print(f"  (Random weights give poor accuracy -- training needed)")

DEMO 2: Circle classification with 2-8-1 network
  Dataset: 200 points (35 inside, 165 outside)
  Accuracy with random weights: 35/200 (17.5%)
  Parameters: 33
  (Random weights give poor accuracy -- training needed)


In [73]:
print("=" * 60)
print("DEMO 3: Forward pass internals on XOR")
print("=" * 60)

for inputs, expected in xor_data:
    xor_net.forward(inputs)
    h = xor_net.layers[0].last_output
    o = xor_net.layers[1].last_output
    print(f"  Input: {inputs}")
    print(f"    Hidden: [{h[0]:.6f}, {h[1]:.6f}]")
    print(f"    Output: {o[0]:.6f} -> {'1' if o[0] >= 0.5 else '0'} (expected: {expected})")


DEMO 3: Forward pass internals on XOR
  Input: [0, 0]
    Hidden: [0.000045, 1.000000]
    Output: 0.000045 -> 0 (expected: 0)
  Input: [0, 1]
    Hidden: [0.999955, 0.999955]
    Output: 0.999955 -> 1 (expected: 1)
  Input: [1, 0]
    Hidden: [0.999955, 0.999955]
    Output: 0.999955 -> 1 (expected: 1)
  Input: [1, 1]
    Hidden: [1.000000, 0.000045]
    Output: 0.000045 -> 0 (expected: 0)


In [74]:
print("=" * 60)
print("DEMO 4: Parameter count for classic architectures")
print("=" * 60)

architectures = [
    ("2-3-1 (this lesson)", [2, 3, 1]),
    ("2-8-1 (circle)", [2, 8, 1]),
    ("784-256-128-10 (MNIST)", [784, 256, 128, 10]),
    ("784-512-256-128-10 (deep MNIST)", [784, 512, 256, 128, 10]),
]

for name, sizes in architectures:
    layers = []
    for i in range(1, len(sizes)):
        layers.append(Layer(n_inputs=sizes[i - 1], n_neurons=sizes[i]))
    net = Network(layers)
    print(f"  {name}: {net.count_parameters():,} parameters")

DEMO 4: Parameter count for classic architectures
  2-3-1 (this lesson): 13 parameters
  2-8-1 (circle): 33 parameters
  784-256-128-10 (MNIST): 235,146 parameters
  784-512-256-128-10 (deep MNIST): 567,434 parameters


## Exercises

1. Build a 2-4-2-1 network (two hidden layers) and run the forward pass on XOR data with random weights. Print the intermediate hidden layer outputs to see how the representation transforms at each layer.

2. Change the hidden layer size in the circle classifier from 8 to 2, then to 32. Run the forward pass with random weights each time. Does the number of hidden neurons change the output range or distribution? Why?

3. Implement a `count_parameters` method on the Network class that returns the total number of trainable weights and biases. Test it on a 784-256-128-10 network (the classic MNIST architecture). How many parameters does it have?

4. Build a forward pass for a 3-4-4-2 network. Feed it RGB color values (normalized to 0-1) and observe the two outputs. This is the architecture for a simple color classifier with two classes.

5. Replace sigmoid with a "leaky step" function: return 0.01 * z if z < 0, else 1.0. Run the forward pass on XOR with the same hand-tuned weights from Step 4. Does it still work? Why is the smooth sigmoid preferred over hard cutoffs?


In [117]:
print("=" * 60)
print("Exercise 1: XOR with hand-tuned 2-4-2-1 network")
print("=" * 60)

# Construyendo una red neuronal con 2 capas ocultas para resolver el problema XOR
# Primera capa oculta con 4 neuronas, tomando la entrada original de 2 características
hidden_layer_1 = Layer(
    n_inputs=2,
    n_neurons=4
)

# Segunda capa oculta con 2 neuronas, tomando la salida de la primera capa como entrada
hidden_layer_2 = Layer(
    n_inputs=4,  
    n_neurons=2
)

# Capa de salida con 1 neurona, tomando la salida de la segunda capa como entrada
output = Layer(
    n_inputs=2,
    n_neurons=1
)

# Creando la red neuronal con las capas definidas
xor_net = Network([hidden_layer_1, hidden_layer_2, output])

# Datos de entrada y salida esperada para el problema XOR
xor_data = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]


random.seed(42)

contador = 1

for inputs, expected in xor_data:
    result = xor_net.forward( inputs )
    print(f'Resultado: { result[0] }')
    predicted = 1 if result[0] >= 0.5 else 0
    status = "Correcto" if predicted == expected else "Incorrecto"
    for layer in xor_net.layers:
        
        if contador >= len(xor_net.layers):
            print(f'Output: {layer.last_output[0]:.5f} -> { predicted  }  (expected: { expected } = { status } )')
        else:
            print(f'Hidden: { contador }: {layer.last_output}')
        contador += 1
    contador = 1

    print()


Exercise 1: XOR with hand-tuned 2-4-2-1 network
Resultado: 0.4734348362326573
Hidden: 1: [0.5, 0.5, 0.5, 0.5]
Hidden: 2: [0.3048542593878748, 0.35897850882197274]
Output: 0.47343 -> 0  (expected: 0 = Correcto )

Resultado: 0.4732888941088188
Hidden: 1: [0.27888914792203123, 0.3650345633564037, 0.5874415780754889, 0.3044656058311966]
Hidden: 2: [0.32868528248562373, 0.43033259065898866]
Output: 0.47329 -> 0  (expected: 1 = Incorrecto )

Resultado: 0.4730894663834834
Hidden: 1: [0.56926514557087, 0.38937470755111603, 0.6160799528479489, 0.6866188461219183]
Hidden: 2: [0.31122609272530166, 0.3711733977423654]
Output: 0.47309 -> 0  (expected: 1 = Incorrecto )

Resultado: 0.4731424267598244
Hidden: 1: [0.3382450042948596, 0.2682499326909251, 0.6955807834931, 0.48956071744213897]
Hidden: 2: [0.3337529729566705, 0.4429123707194635]
Output: 0.47314 -> 0  (expected: 0 = Correcto )



In [145]:
print("=" * 60)
print("Exercise 2: Circle classification with 2-2-1 network")
print("=" * 60)

random.seed(42)

data = []
for _ in range(200):
    x = random.uniform(-1, 1)
    y = random.uniform(-1, 1)
    label = 1 if (x * x + y * y) < 0.25 else 0
    data.append(([x, y], label))

inside_count = sum(1 for _, label in data if label == 1)
outside_count = len(data) - inside_count
print(f"  Dataset: {len(data)} points ({inside_count} inside, {outside_count} outside)")

random.seed(7)
circle_net = Network([
    Layer(n_inputs=2, n_neurons=2),
    Layer(n_inputs=2, n_neurons=1),
])

correct = 0
for inputs, expected in data:
    result = circle_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    if predicted == expected:
        correct += 1

print(f"  Accuracy with random weights: {correct}/{len(data)} ({100 * correct / len(data):.1f}%)")
print(f"  Parameters: {circle_net.count_parameters()}")
print(f"  (Random weights give poor accuracy -- training needed)")

print()
print("=" * 60)
print("Exercise 2: Circle classification with 2-32-1 network")
print("=" * 60)

print(f"  Dataset: {len(data)} points ({inside_count} inside, {outside_count} outside)")

circle_net = Network([
    Layer(n_inputs=2, n_neurons=32),
    Layer(n_inputs=32, n_neurons=1),
])

correct = 0
for inputs, expected in data:
    result = circle_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    if predicted == expected:
        correct += 1

print(f"  Accuracy with random weights: {correct}/{len(data)} ({100 * correct / len(data):.1f}%)")
print(f"  Parameters: {circle_net.count_parameters()}")
print(f"  (Random weights give poor accuracy -- training needed)")


print('\n\nDoes the number of hidden neurons change the output range or distribution? Why?')
print('No, el número de neuronas ocultas no cambia el rango de salida del clasificador.')
print('La salida final siempre pasa por la función sigmoide, por lo que cada valor está entre 0 y 1.')

Exercise 2: Circle classification with 2-2-1 network
  Dataset: 200 points (35 inside, 165 outside)
  Accuracy with random weights: 165/200 (82.5%)
  Parameters: 9
  (Random weights give poor accuracy -- training needed)

Exercise 2: Circle classification with 2-32-1 network
  Dataset: 200 points (35 inside, 165 outside)
  Accuracy with random weights: 165/200 (82.5%)
  Parameters: 129
  (Random weights give poor accuracy -- training needed)


Does the number of hidden neurons change the output range or distribution? Why?
No, el número de neuronas ocultas no cambia el rango de salida del clasificador.
La salida final siempre pasa por la función sigmoide, por lo que cada valor está entre 0 y 1.


In [118]:
print("=" * 60)
print("Exercise 4: Color classifier with 3-4-4-2 network")
print("=" * 60)

random.seed(42)
color_net = Network([
    Layer(n_inputs=3, n_neurons=4),
    Layer(n_inputs=4, n_neurons=4),
    Layer(n_inputs=4, n_neurons=2),
])

color_samples = [
    ([1.0, 0.0, 0.0], "red"),
    ([0.0, 1.0, 0.0], "green"),
    ([0.0, 0.0, 1.0], "blue"),
    ([1.0, 1.0, 0.0], "yellow"),
    ([0.0, 1.0, 1.0], "cyan"),
]

for inputs, label in color_samples:
    output = color_net.forward(inputs)
    predicted_class = 0 if output[0] >= output[1] else 1
    print(f"  {label:8s} {inputs} -> [{output[0]:.4f}, {output[1]:.4f}] predicted_class={predicted_class}")


Exercise 4: Color classifier with 3-4-4-2 network
  red      [1.0, 0.0, 0.0] -> [0.7030, 0.5965] predicted_class=0
  green    [0.0, 1.0, 0.0] -> [0.6996, 0.5899] predicted_class=0
  blue     [0.0, 0.0, 1.0] -> [0.6916, 0.5843] predicted_class=0
  yellow   [1.0, 1.0, 0.0] -> [0.7081, 0.6034] predicted_class=0
  cyan     [0.0, 1.0, 1.0] -> [0.6987, 0.5918] predicted_class=0


In [ ]:
print("=" * 60)
print("Exercise 5: XOR with hand-tuned 2-2-1 network")
print("=" * 60)

# Definiendo una función de activación leaky step para permitir valores negativos en la salida
def leaky_step(x):
    return 1.0 if x >= 0 else 0.01 * x

# Definiendo la clase Layer para representar una capa de la red neuronal
class Layer_B:
    # Inicializando la capa con el número de entradas, neuronas, pesos y sesgos
    def __init__(self, n_inputs, n_neurons, weights=None, biases=None):
        # Si se proporcionan pesos, se utilizan, de lo contrario se generan aleatoriamente
        if weights is not None:
            self.weights = weights
        else:
            self.weights = [
                [random.uniform(-1, 1) for _ in range(n_inputs)]
                for _ in range(n_neurons)
            ]
        # Si se proporcionan sesgos, se utilizan, de lo contrario se inicializan a cero por cada neurona
        if biases is not None:
            self.biases = biases
        else:
            self.biases = [0.0] * n_neurons

    # Definiendo el método forward para calcular la salida de la capa dada una entrada
    def forward(self, inputs):
        # Guardando la entrada para su uso posterior
        self.last_input = inputs
        # Definiendo la salida de cada neurona en la capa
        self.last_output = []
        # Iterando sobre cada neurona para calcular su salida
        for neuron_idx in range(len(self.weights)):
            # Calculando la suma ponderada de las entradas para la neurona actual
            z = sum(
                w * x for w, x in zip(self.weights[neuron_idx], inputs)
            )
            # Agregando el sesgo a la suma ponderada
            z += self.biases[neuron_idx]
            # Aplicando la función de activación leaky step a la suma ponderada y agregando el resultado a la salida de la capa
            self.last_output.append(leaky_step(z))
        # Retornando la salida de la capa
        return self.last_output
    

hidden = Layer_B(
    n_inputs=2,
    n_neurons=2,
    weights=[[20.0, 20.0], [-20.0, -20.0]],
    biases=[-10.0, 30.0],
)

output = Layer_B(
    n_inputs=2,
    n_neurons=1,
    weights=[[20.0, 20.0]],
    biases=[-30.0],
)

xor_net = Network([hidden, output])

xor_data = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

contador = 1

for inputs, expected in xor_data:
    result = xor_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    #print(f"  {inputs} -> {result[0]:.6f} (rounded: {predicted}, expected: {expected})")

    status = "Correcto" if predicted == expected else "Incorrecto"
    for layer in xor_net.layers:
        
        if contador >= len(xor_net.layers):
            print(f'Output: {layer.last_output[0]:.5f} -> { predicted  }  (expected: { expected } = { status } )')
        else:
            print(f'Hidden: { contador }: {layer.last_output}')
        contador += 1
    contador = 1
    print()

Exercise 5: XOR with hand-tuned 2-2-1 network
Hidden: 1: [-0.1, 1.0]
Output: -0.12000 -> 0  (expected: 0 = Correcto )

Hidden: 1: [1.0, 1.0]
Output: 1.00000 -> 1  (expected: 1 = Correcto )

Hidden: 1: [1.0, 1.0]
Output: 1.00000 -> 1  (expected: 1 = Correcto )

Hidden: 1: [1.0, -0.1]
Output: -0.12000 -> 0  (expected: 0 = Correcto )

